# INST447 — Week 2: Pandas — Inspect, Filter, Transform

**Course:** INST447: Data Sources and Manipulation  
**Lecture:** Week 2, Fall 2026  
**Topic:** Pandas: Inspect, Filter, Transform

This notebook turns the Week 2 lecture into study notes plus runnable examples.

## Learning goals

By the end of this notebook, you should be able to:

- explain what rows, columns, cells, and shape mean in a pandas `DataFrame`;
- inspect a DataFrame with `head()`, `tail()`, `shape`, `columns`, `dtypes`, and `index`;
- select rows and columns with brackets, `.loc[]`, and `.iloc[]`;
- filter rows with Boolean masks, `.isin()`, `.isna()`, and combined conditions;
- transform columns with arithmetic, `.astype()`, and `.str`;
- create new columns from existing columns;
- use functions, lambda expressions, and `.apply()`;
- understand the difference between `axis=0` and `axis=1` in `DataFrame.apply()`;
- convert strings to datetimes and extract weekday information;
- sort rows and read pandas method chains;
- recognize how notebook state and execution order can change results.

> **Core habit from the lecture:** run a small operation, inspect what it produced, then continue.


## 1. Flight data and DataFrames

The lecture uses a small personal flight log. Each **row** represents one flight. Each **column** records one kind of information about that flight, and each **cell** is one value at the intersection of a row and a column.

A DataFrame is two-dimensional, but it is not simply a matrix with prettier labels. A table can contain columns with different meanings and data types, such as dates, airport codes, distances, prices, and delays.


In [1]:
import pandas as pd

flights_data = [
    ("2024-01-15", "UA1247", "BWI", "ORD", "651", "B737", "12A", 289.50, 15),
    ("2024-01-22", "DL456", "ORD", "LAX", "1745", "A321", "8F", 425.00, None),
    ("2024-02-08", "WN2891", "LAX", "PHX", "370", "B737", "", 149.99, 0),
    ("2024-02-10", "WN1055", "PHX", "DEN", "602", "B737", "15C", None, 45),
    ("2024-03-05", "AA892", "DEN", "DFW", "663", "B737", "21B", 198.75, None),
    ("2024-03-12", "UA634", "DFW", "IAD", "1216", "B777", "9A", 345.25, 12),
    ("2024-04-20", "B61840", "IAD", "BOS", "429", "", "11D", 179.50, 0),
    ("2024-05-15", "DL1123", "BOS", "ATL", "946", "A220", "4A", 267.00, 25),
    ("2024-05-18", "DL2967", "ATL", "MIA", "594", "B737", "", None, 8),
    ("2024-06-02", "AA1456", "MIA", "LGA", "1095", "A321", "18F", 312.80, None),
]

columns = [
    "flight_date",
    "flight_number",
    "origin",
    "destination",
    "distance",
    "aircraft",
    "seat",
    "price",
    "delay_min",
]

flights = pd.DataFrame(flights_data, columns=columns)
flights

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.00,NaN
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0
4,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN
5,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0
6,2024-04-20,B61840,IAD,BOS,429,,11D,179.50,0.0
7,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.00,25.0
8,2024-05-18,DL2967,ATL,MIA,594,B737,,NaN,8.0
9,2024-06-02,AA1456,MIA,LGA,1095,A321,18F,312.80,NaN


### Questions this table can answer

The lecture uses this dataset to motivate common data-manipulation tasks:

- find flights from or to a particular airport;
- compare distance, price, and delay;
- make a route label such as `BWI-ORD`;
- find the longest trips;
- add the day of the week.

The important design question is always: **what does one row represent?**


In [2]:
print("Shape:", flights.shape)
print("Rows:", flights.shape[0])
print("Columns:", flights.shape[1])

Shape: (10, 9)
Rows: 10
Columns: 9


## 2. Inspecting a DataFrame

Before filtering or transforming data, inspect it. Useful commands from the lecture include:

- `df.head()` — first rows;
- `df.tail(n)` — last `n` rows;
- `df.shape` — `(number_of_rows, number_of_columns)`;
- `df.columns` — column labels;
- `df.dtypes` — data type of each column;
- `df.index` — row labels.

This is the "look before you manipulate" part of the workflow.


In [3]:
flights.head()

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.00,NaN
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0
4,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN


In [4]:
flights.tail(3)

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
7,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.0,25.0
8,2024-05-18,DL2967,ATL,MIA,594,B737,,NaN,8.0
9,2024-06-02,AA1456,MIA,LGA,1095,A321,18F,312.8,NaN


In [5]:
print("Columns:", list(flights.columns))
print("\nData types:")
print(flights.dtypes)
print("\nIndex:", flights.index)

Columns: ['flight_date', 'flight_number', 'origin', 'destination', 'distance', 'aircraft', 'seat', 'price', 'delay_min']

Data types:
flight_date       object
flight_number     object
origin            object
destination       object
distance          object
aircraft          object
seat              object
price            float64
delay_min        float64
dtype: object

Index: RangeIndex(start=0, stop=10, step=1)


## 3. Selecting rows: `iloc` vs. `loc`

### `iloc`
`iloc` uses **integer positions**. Python-style slicing excludes the ending position.

```python
flights.iloc[2:5]
```

This returns positions 2, 3, and 4.

### `loc`
`loc` uses **row labels**. With the default integer labels, the ending label is included.

```python
flights.loc[2:5]
```

This returns rows labeled 2, 3, 4, and 5.

That inclusive/exclusive difference is a classic pandas gotcha.


In [6]:
flights.iloc[2:5]

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0
4,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN


In [7]:
flights.loc[2:5]

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0
4,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN
5,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0


### Changing the index

The index contains the labels pandas uses for rows. `set_index()` can move a column into the row labels. `reset_index()` can move it back into a normal column.


In [8]:
by_number = flights.set_index("flight_number")
by_number.head(3)

,flight_date,origin,destination,distance,aircraft,seat,price,delay_min
flight_number,,,,,,,,
UA1247,2024-01-15,BWI,ORD,651,B737,12A,289.50,15.0
DL456,2024-01-22,ORD,LAX,1745,A321,8F,425.00,NaN
WN2891,2024-02-08,LAX,PHX,370,B737,,149.99,0.0


## 4. Selecting columns

A single column can be selected with dot notation or bracket notation:

```python
flights.origin
flights["origin"]
```

Bracket notation is more general and avoids conflicts with built-in DataFrame attributes. For example, a DataFrame can have a column literally named `size`, while `df.size` already means the total number of cells.

For **multiple columns**, use a list inside the brackets:

```python
flights[["flight_number", "origin", "destination"]]
```


In [9]:
flights["origin"].head(3)

0    BWI
1    ORD
2    LAX
Name: origin, dtype: object

In [10]:
flights[["flight_number", "origin", "destination"]].head()

,flight_number,origin,destination
0,UA1247,BWI,ORD
1,DL456,ORD,LAX
2,WN2891,LAX,PHX
3,WN1055,PHX,DEN
4,AA892,DEN,DFW


In [11]:
weird_df = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "size": [11, 12, 13, 14]
})

print("Column named size:")
print(weird_df["size"])

print("\nDataFrame.size:")
print(weird_df.size)

Column named size:
0    11
1    12
2    13
3    14
Name: size, dtype: int64

DataFrame.size:
8


### Series vs. one-column DataFrame

These contain the same values but are different pandas objects:

- `flights["origin"]` → one-dimensional `Series`, shape `(10,)`
- `flights[["origin"]]` → two-dimensional `DataFrame`, shape `(10, 1)`


In [12]:
print("Series shape:", flights["origin"].shape)
print("One-column DataFrame shape:", flights[["origin"]].shape)

Series shape: (10,)
One-column DataFrame shape: (10, 1)


### Selecting rows and columns together with `loc`

`loc` accepts two selectors:

```python
df.loc[row_selector, column_selector]
```

The first position chooses rows; the second chooses columns.


In [13]:
flights.loc[
    2:5,
    ["origin", "destination"],
]

,origin,destination
2,LAX,PHX
3,PHX,DEN
4,DEN,DFW
5,DFW,IAD


## 5. Filtering DataFrames with Boolean conditions

Filtering begins with a condition evaluated once per row.

```python
flights.origin == "BWI"
```

The result is a Boolean `Series`: one `True` or `False` value for each row. Put that Boolean Series inside selection brackets to keep only the `True` rows.


In [14]:
flights.origin == "BWI" 

0     True
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
Name: origin, dtype: bool

In [15]:
flights[flights.origin == "BWI"]

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.5,15.0


### Membership tests with `.isin()`

Use `.isin(collection)` when you want to ask whether each value belongs to a set/list of acceptable values.


In [16]:
east_coast = ["BWI", "BOS", "LGA", "IAD"]
flights[flights.origin.isin(east_coast)]

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.5,15.0
6,2024-04-20,B61840,IAD,BOS,429,,11D,179.5,0.0
7,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.0,25.0


### Missing values: `.isna()` is not the same as an empty string

The lecture deliberately distinguishes:

- a pandas missing value, such as `None`/`NaN`;
- an empty string, `""`, which is still a stored string value.

So `isna()` will **not** detect an empty string.


In [17]:
print("Rows where seat is missing according to pandas:")
display(flights[flights.seat.isna()])

print("Rows where seat is an empty string:")
display(flights[flights.seat == ""])

Rows where seat is missing according to pandas:


,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min


Rows where seat is an empty string:


,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0
8,2024-05-18,DL2967,ATL,MIA,594,B737,,NaN,8.0


### Combining Boolean conditions

Use:

- `&` for elementwise **and**
- `|` for elementwise **or**

Put each comparison inside parentheses.

```python
flights[(condition_1) & (condition_2)]
```


In [18]:
flights[(flights.price > 200) & (flights.price.notna())]

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.00,NaN
5,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0
7,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.00,25.0
9,2024-06-02,AA1456,MIA,LGA,1095,A321,18F,312.80,NaN


### What brackets can select

The contents of brackets change the operation:

```python
flights["origin"]                  # one column
flights[["origin", "price"]]       # multiple columns
flights[flights.price > 200]       # rows selected by Boolean mask
```

This overloading is why it helps to read pandas expressions carefully.


### Filter rows and choose columns with `loc`

A Boolean condition can be used as the row selector in `.loc[]`.


In [19]:
flights.loc[
    flights.destination == "LAX",
    ["origin", "destination"],
]

,origin,destination
1,ORD,LAX


### Filtering with `query()`

`query()` expresses the condition as a string. It can be easier to read for some filters, but it is **not SQL**.


In [20]:
flights.query("origin == 'BWI'")

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.5,15.0


In [21]:
flights.query("price > 300 and origin == 'ORD'")

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.0,NaN


## 6. Transforming columns

Pandas often applies one operation to an entire Series at once. This is sometimes called a **vectorized** operation.

The lecture's key example is the `distance` column. It looks numeric, but it was originally stored as strings. That means multiplication behaves like Python string repetition rather than numeric multiplication.


In [22]:
# Re-create the original distance column as strings so the behavior is obvious.
demo_distance = pd.Series(["651", "1745", "370"])

print("String multiplication:")
print(demo_distance * 2)

String multiplication:
0      651651
1    17451745
2      370370
dtype: object


### Data types explain behavior

Before converting, the flight `distance` values are strings while `price` values are numeric. Operators follow the underlying data type.


In [23]:
# Reset the main DataFrame so this section starts from the lecture's original state.
flights = pd.DataFrame(flights_data, columns=columns)
flights[["distance", "price"]].dtypes

distance     object
price       float64
dtype: object

In [24]:
flights.distance * 2

0      651651
1    17451745
2      370370
3      602602
4      663663
5    12161216
6      429429
7      946946
8      594594
9    10951095
Name: distance, dtype: object

In [25]:
flights.price * 2

0    579.00
1    850.00
2    299.98
3       NaN
4    397.50
5    690.50
6    359.00
7    534.00
8       NaN
9    625.60
Name: price, dtype: float64

### `.astype()` returns converted values — assignment saves them

This previews a conversion:

```python
flights.distance.astype(int)
```

But simply evaluating that expression does **not** mutate `flights`. To save it:

```python
flights["distance"] = flights.distance.astype(int)
```


In [26]:
flights.distance.astype(int).head()

0     651
1    1745
2     370
3     602
4     663
Name: distance, dtype: int64

In [27]:
flights["distance"] = flights.distance.astype(int)
print("New dtype:", flights.distance.dtype)
print("\nRound-trip distance:")
flights.distance * 2

New dtype: int64

Round-trip distance:


0    1302
1    3490
2     740
3    1204
4    1326
5    2432
6     858
7    1892
8    1188
9    2190
Name: distance, dtype: int64

## 7. Notebook state and execution order

Jupyter notebooks keep the current in-memory version of objects such as `flights`, and cells can be run in almost any order.

That means:

- a mutation in a later cell can change what an earlier cell produces when you rerun it;
- your code may appear to work only because of the current notebook state;
- restarting the kernel and using **Run All** is a strong reproducibility check;
- for reusable data processing, keep the raw input and explicitly record each transformation.

The lecture connects this to the Week 1 pipeline/DAG idea: input → cleaning/validation → transformation → analysis → outputs.


## 8. Creating new columns

Pandas aligns values by row, so you can combine Series to create new information.


In [28]:
flights["route"] = flights.origin + "-" + flights.destination
flights[["origin", "destination", "route"]].head()

,origin,destination,route
0,BWI,ORD,BWI-ORD
1,ORD,LAX,ORD-LAX
2,LAX,PHX,LAX-PHX
3,PHX,DEN,PHX-DEN
4,DEN,DFW,DEN-DFW


In [29]:
# A derived numeric quantity.
# Rows with missing price naturally produce NaN.
flights["miles_per_dollar"] = flights.distance / flights.price
flights[["flight_number", "distance", "price", "miles_per_dollar"]].head()

,flight_number,distance,price,miles_per_dollar
0,UA1247,651,289.50,2.248705
1,DL456,1745,425.00,4.105882
2,WN2891,370,149.99,2.466831
3,WN1055,602,NaN,NaN
4,AA892,663,198.75,3.335849


## 9. String transformations with `.str`

A Python slice such as `"UA1247"[:2]` works on one string.

But `flights.flight_number[:2]` slices the **Series rows**, not each string. To apply string operations to every value in a Series, use the pandas `.str` accessor.


In [30]:
print("One Python string:")
print("UA1247"[:2])

print("\nFirst two rows of the Series — not first two characters of every string:")
display(flights.flight_number[:2])

print("First two characters of every flight number:")
display(flights.flight_number.str[:2])

One Python string:
UA

First two rows of the Series — not first two characters of every string:


0    UA1247
1     DL456
Name: flight_number, dtype: object

First two characters of every flight number:


0    UA
1    DL
2    WN
3    WN
4    AA
5    UA
6    B6
7    DL
8    DL
9    AA
Name: flight_number, dtype: object

In [31]:
print("Uppercase aircraft codes:")
display(flights.aircraft.str.upper())

print("Does aircraft text contain 'B7'?")
display(flights.aircraft.str.contains("B7"))

Uppercase aircraft codes:


0    B737
1    A321
2    B737
3    B737
4    B737
5    B777
6        
7    A220
8    B737
9    A321
Name: aircraft, dtype: object

Does aircraft text contain 'B7'?


0     True
1    False
2     True
3     True
4     True
5     True
6    False
7    False
8     True
9    False
Name: aircraft, dtype: bool

## 10. Functions, lambda expressions, and `apply()`

`apply()` is useful when you want pandas to call Python logic repeatedly.

### Row-wise `apply`

To combine `origin` and `destination` using Python's `"-".join`, pass the function to `apply()` and use `axis=1`.


In [32]:
flights[["origin", "destination"]].apply(
    "-".join,
    axis=1,
)

0    BWI-ORD
1    ORD-LAX
2    LAX-PHX
3    PHX-DEN
4    DEN-DFW
5    DFW-IAD
6    IAD-BOS
7    BOS-ATL
8    ATL-MIA
9    MIA-LGA
dtype: object

### Lambda on a column

A lambda is a short unnamed function.

```python
lambda x: x * 1.6
```

Here:

- `x` names one input value;
- `x * 1.6` is the returned value.


In [33]:
flights.distance.apply(lambda x: x * 1.6)

0    1041.6
1    2792.0
2     592.0
3     963.2
4    1060.8
5    1945.6
6     686.4
7    1513.6
8     950.4
9    1752.0
Name: distance, dtype: float64

### Named functions

If the logic is reusable or needs a descriptive name, define a normal function and pass the function itself to `apply()`.


In [34]:
def tokm(x):
    return x * 1.6

flights.distance.apply(tokm)

0    1041.6
1    2792.0
2     592.0
3     963.2
4    1060.8
5    1945.6
6     686.4
7    1513.6
8     950.4
9    1752.0
Name: distance, dtype: float64

### Categorizing distances

The lecture categorizes flights as `Short`, `Medium`, or `Long` using conditional logic.


In [35]:
flights["flight_type"] = flights.distance.apply(
    lambda x: (
        "Short" if x < 500
        else "Medium" if x < 1000
        else "Long"
    )
)

flights[["flight_number", "distance", "flight_type"]]

,flight_number,distance,flight_type
0,UA1247,651,Medium
1,DL456,1745,Long
2,WN2891,370,Short
3,WN1055,602,Medium
4,AA892,663,Medium
5,UA634,1216,Long
6,B61840,429,Short
7,DL1123,946,Medium
8,DL2967,594,Medium
9,AA1456,1095,Long


### `DataFrame.apply()` and the `axis` gotcha

By default, `DataFrame.apply()` uses `axis=0`, so the function receives **one column at a time**.

If your function is written as though its input is a row, a call without `axis=1` can fail with a `KeyError`.

Use:

```python
flights.apply(your_row_function, axis=1)
```

when the function needs fields from each row.


In [36]:
flights.apply(
    lambda row: (
        f"Flight {row['flight_number']} "
        f"from {row['origin']} to {row['destination']}"
    ),
    axis=1,
)

0    Flight UA1247 from BWI to ORD
1     Flight DL456 from ORD to LAX
2    Flight WN2891 from LAX to PHX
3    Flight WN1055 from PHX to DEN
4     Flight AA892 from DEN to DFW
5     Flight UA634 from DFW to IAD
6    Flight B61840 from IAD to BOS
7    Flight DL1123 from BOS to ATL
8    Flight DL2967 from ATL to MIA
9    Flight AA1456 from MIA to LGA
dtype: object

### A function with several conditions

A named function can be easier to read than a large lambda when several branches are involved.


In [37]:
def flight_experience(row):
    if pd.isna(row["delay_min"]) or row["delay_min"] == 0:
        delay_status = "On time"
    elif row["delay_min"] < 30:
        delay_status = "Minor delay"
    else:
        delay_status = "Major delay"

    airline_code = row["flight_number"][:2]
    return f"{airline_code} flight: {delay_status}"

flights["experience"] = flights.apply(flight_experience, axis=1)
flights[["flight_number", "delay_min", "experience"]].head(6)

,flight_number,delay_min,experience
0,UA1247,15.0,UA flight: Minor delay
1,DL456,NaN,DL flight: On time
2,WN2891,0.0,WN flight: On time
3,WN1055,45.0,WN flight: Major delay
4,AA892,NaN,AA flight: On time
5,UA634,12.0,UA flight: Minor delay


## 11. Datetime conversion and weekday names

String dates can be converted to pandas datetime values with `pd.to_datetime()`.

After conversion, the `.dt` accessor provides date/time operations:

- `.dt.day_name()` → weekday name;
- `.dt.dayofweek` → Monday = 0 through Sunday = 6;
- `>= 5` therefore identifies Saturday/Sunday.


In [38]:
dates = pd.to_datetime(
    flights["flight_date"],
    format="%Y-%m-%d",
)

flights["weekday"] = dates.dt.day_name()
flights["weekend"] = dates.dt.dayofweek >= 5

flights[
    ["flight_date", "flight_number", "weekday", "weekend"]
].head(4)

,flight_date,flight_number,weekday,weekend
0,2024-01-15,UA1247,Monday,False
1,2024-01-22,DL456,Monday,False
2,2024-02-08,WN2891,Thursday,False
3,2024-02-10,WN1055,Saturday,True


## 12. Sorting and selecting extremes

Use `sort_values()` when you want a complete ordering.

Use `nlargest()` or `nsmallest()` when you directly want only the top or bottom few rows.


In [39]:
flights.sort_values(
    "distance",
    ascending=False,
)[["flight_number", "route", "distance"]].head(5)

,flight_number,route,distance
1,DL456,ORD-LAX,1745
5,UA634,DFW-IAD,1216
9,AA1456,MIA-LGA,1095
7,DL1123,BOS-ATL,946
4,AA892,DEN-DFW,663


In [40]:
flights.nlargest(
    3,
    "distance",
)[["flight_number", "route", "distance"]]

,flight_number,route,distance
1,DL456,ORD-LAX,1745
5,UA634,DFW-IAD,1216
9,AA1456,MIA-LGA,1095


## 13. Reading a pandas method chain

The lecture asks you to read a chain from top to bottom as a sequence of operations:

1. keep rows with known prices;
2. sort by price from high to low;
3. keep the first three rows.

The point is not just shorter code. Each method represents one understandable transformation in the workflow.


In [41]:
chain_data = flights.loc[:4, ["flight_number", "origin", "price"]].copy()
chain_data

,flight_number,origin,price
0,UA1247,BWI,289.50
1,DL456,ORD,425.00
2,WN2891,LAX,149.99
3,WN1055,PHX,NaN
4,AA892,DEN,198.75


In [42]:
chain_data.loc[
    chain_data["price"].notna()
].sort_values(
    "price", ascending=False
).head(3)

,flight_number,origin,price
1,DL456,ORD,425.00
0,UA1247,BWI,289.50
4,AA892,DEN,198.75


## 14. Week 2 cheat sheet

### Look
```python
df.head()
df.tail(3)
df.shape
df.columns
df.dtypes
df.index
```

### Select
```python
df["column"]
df[["col1", "col2"]]
df.iloc[2:5]
df.loc[2:5]
df.loc[row_selector, column_selector]
```

### Filter
```python
df[df["col"] == value]
df[df["col"].isin(values)]
df[df["col"].isna()]
df[df["col"].notna()]
df[(condition1) & (condition2)]
df[(condition1) | (condition2)]
df.query("price > 300")
```

### Transform
```python
df["numeric"] * 2
df["numeric"] = df["numeric"].astype(int)
df["new"] = df["a"] + "-" + df["b"]
df["text"].str[:2]
df["text"].str.upper()
df["text"].str.contains("B7")
```

### Functions and `apply`
```python
series.apply(function)
series.apply(lambda x: ...)
df.apply(function, axis=1)
```

### Dates
```python
dates = pd.to_datetime(df["date"], format="%Y-%m-%d")
dates.dt.day_name()
dates.dt.dayofweek
```

### Sort / top values
```python
df.sort_values("col", ascending=False)
df.nlargest(3, "col")
df.nsmallest(3, "col")
```


## 15. Practice

Try these without looking at the solutions first.

1. Show only flights whose destination is `DEN`.
2. Show only `flight_number`, `origin`, and `price` for flights with a known price above 250.
3. Create an `airline` column from the first two characters of `flight_number`.
4. Create a Boolean column named `long_flight` that is `True` when distance is at least 1000 miles.
5. Find the two cheapest flights with known prices.
6. Create a sentence for every row such as `"UA1247: BWI to ORD"` using row-wise `apply()`.


In [43]:
# Practice workspace
# Write your answers here.


### Practice solutions

Open this section after you have tried the questions.


In [44]:
# 1. Destination is DEN
display(flights[flights.destination == "DEN"])

# 2. Selected columns, known price > 250
display(
    flights.loc[
        (flights.price.notna()) & (flights.price > 250),
        ["flight_number", "origin", "price"],
    ]
)

# 3. Airline code
flights["airline"] = flights.flight_number.str[:2]

# 4. Long-flight Boolean
flights["long_flight"] = flights.distance >= 1000

# 5. Two cheapest known-price flights
display(
    flights[flights.price.notna()]
    .nsmallest(2, "price")[["flight_number", "route", "price"]]
)

# 6. Row-wise sentence
sentences = flights.apply(
    lambda row: f"{row['flight_number']}: {row['origin']} to {row['destination']}",
    axis=1,
)
sentences.head()

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min,route,miles_per_dollar,flight_type,experience,weekday,weekend
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0,PHX-DEN,NaN,Medium,WN flight: Major delay,Saturday,True


,flight_number,origin,price
0,UA1247,BWI,289.50
1,DL456,ORD,425.00
5,UA634,DFW,345.25
7,DL1123,BOS,267.00
9,AA1456,MIA,312.80


,flight_number,route,price
2,WN2891,LAX-PHX,149.99
6,B61840,IAD-BOS,179.50


0    UA1247: BWI to ORD
1     DL456: ORD to LAX
2    WN2891: LAX to PHX
3    WN1055: PHX to DEN
4     AA892: DEN to DFW
dtype: object

## Final takeaways

The Week 2 lecture is less about memorizing pandas syntax and more about building a dependable manipulation workflow:

1. **Inspect** the object and its types.
2. **Select** the rows/columns you need.
3. **Filter** with explicit conditions.
4. **Transform** columns while paying attention to data types.
5. Use `.str`, datetime accessors, functions, and `apply()` when the operation requires them.
6. **Inspect intermediate results** instead of writing a giant transformation all at once.
7. Remember that notebook state matters; restart and **Run All** to verify the workflow from the beginning.

That pattern scales into later INST447 topics such as grouping, joining, reshaping, and larger data pipelines.
